# Smart Classroom - Step 6: train PhysNet (3D CNN) on UBFC-rPPG and compare with POS

1. **Runtime -> Change runtime type -> T4 GPU**
2. UBFC-rPPG **DATASET_2** must be in your Google Drive (folder with `subject1/vid.avi`, `subject1/ground_truth.txt`, ...).
3. Your latest code must be pushed to GitHub (`git push`): cell 2 clones it.
4. Run the cells in order. Preprocessing is saved to Drive, so it runs only once (already done: cell 3 just says `cached`).
5. Cell 4 evaluates **PhysNet v1** (trained without webcam augmentation) and trains **PhysNet v2** (with webcam augmentation, the fix for the live domain shift).
6. Download `physnet_results.zip` and unzip it **into** your `SmartClassroom` folder (replace the files).

In [ ]:
# 1. Google Drive + paths  (EDIT UBFC if your folder has another name)
from google.colab import drive
drive.mount('/content/drive')
UBFC = '/content/drive/MyDrive/DATASET_2'           # shortcut to DATASET_2 (contains subject1, subject3, ...)
CACHE = '/content/drive/MyDrive/ubfc_cache'         # preprocessed subjects (~30 MB each), kept on Drive
import os
print(sorted(os.listdir(UBFC))[:5], '...', len(os.listdir(UBFC)), 'entries')

In [ ]:
# 2. Code from GitHub + MediaPipe with ONE OpenCV build
#    (Colab ships opencv-python 5.x; mediapipe adds opencv-contrib-python 4.x: two cv2 builds in one
#     process crash with 'corrupted double-linked list', so keep only opencv-contrib-python)
%cd /content
!rm -rf smartClassroom && git clone -q https://github.com/sofiensgh/smartClassroom.git
%cd /content/smartClassroom
!pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python
!pip install -q mediapipe opencv-contrib-python
!pip install -q --force-reinstall --no-deps opencv-contrib-python   # clean cv2 files after the uninstall
!python -c "import cv2; print('cv2', cv2.__version__)"
!pip list 2>/dev/null | grep -i opencv
import torch; print(torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: change the runtime type!')

In [ ]:
# 3. Preprocess: face crops 72x72 + POS skin colour + ground-truth PPG, one .npz per subject
#    (about 1 min per subject; already-processed subjects are skipped, so you can re-run after a disconnect)
!python -u -m smartclass.rppg.preprocess --ubfc "{UBFC}" --out "{CACHE}"

In [ ]:
# 4. v1 and v2 on the 3 tests (clean / webcam-degraded / resting ~70 BPM), then train PhysNet v3
#    v3 = FREQUENCY augmentation (clips played slower/faster: heart rates 45-150 BPM, UBFC is mostly 85-120)
#       + lighter webcam augmentation. About 40 min on a T4.
!python -u -m smartclass.training.train_physnet --cache "{CACHE}" --eval-only models/physnet_v1.pt
!python -u -m smartclass.training.train_physnet --cache "{CACHE}" --eval-only models/physnet_v2.pt
!python -u -m smartclass.training.train_physnet --cache "{CACHE}" --epochs 40

In [ ]:
# 5. Results: v1 vs v2 robustness, then v2 figures (v1 figures stay in output/heart/v1/)
import numpy as np, pandas as pd
from IPython.display import Image, display
display(pd.read_csv('output/heart/robustness.csv'))
display(pd.read_csv('output/heart/results.csv'))
print(open('output/heart/split.txt').read())
for f in ['physnet_curves', 'scatter', 'bland_altman', 'example_wave']:
    display(Image(f'output/heart/{f}.png'))

# report figure: what the webcam augmentation does to a face crop
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from smartclass.training.rppg_data import load_cache, webcam_degrade
import glob
z = np.load(sorted(glob.glob(f'{CACHE}/subject*.npz'))[0])
clip = z['frames'][:128]
fig = Figure(figsize=(10, 4)); FigureCanvasAgg(fig)
axes = fig.subplots(2, 5)
for j in range(5):
    axes[0, j].imshow(clip[j * 25]); axes[0, j].axis('off')
    axes[1, j].imshow(webcam_degrade(clip, np.random.default_rng(j), fps=(18, 18), jpeg=(50, 50), noise=(3, 3), p_blur=1.0)[j * 25]); axes[1, j].axis('off')
axes[0, 0].set_title('UBFC (clean)', loc='left'); axes[1, 0].set_title('webcam augmentation', loc='left')
fig.tight_layout(); fig.savefig('output/heart/webcam_aug_example.png', dpi=150)
display(Image('output/heart/webcam_aug_example.png'))

In [ ]:
# 6. Download v3: unzip INTO your SmartClassroom folder (replace). It adds models/physnet_v3.pt,
#    output/heart/v3/ and robustness.csv; your physnet.pt and validation.csv on the PC are not touched.
!mkdir -p output/heart/v3 && cp output/heart/results.csv output/heart/windows.csv output/heart/*.png output/heart/v3/ && cp models/physnet.pt models/physnet_v3.pt
!rm -f ../physnet_v3.zip && zip -qr ../physnet_v3.zip models/physnet_v3.pt output/heart/v3 output/heart/robustness.csv
from google.colab import files
files.download('../physnet_v3.zip')